In [16]:
# ============================================================
# MODELO CHURN MEJORADO CONTROLADO
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

from imblearn.over_sampling import SMOTE

# ============================================================
# COPIA DEL DATASET
# ============================================================

df = df_model.copy()

# ============================================================
# VARIABLES DERIVADAS
# ============================================================

df["cliente_moroso"] = np.where(df["dias_retraso_pago"] > 0, 1, 0)

df["riesgo_financiero"] = (
    df["impagos_3m"].fillna(0) * 5
    + df["media_retraso_3m"].fillna(0)
)

df["riesgo_consumo"] = (
    df["variacion_consumo_pct"].fillna(0).abs()
    + df["cambio_consumo"].fillna(0).abs()
)

df["subida_factura_flag"] = np.where(df["subida_factura"] > 10, 1, 0)

# ============================================================
# VARIABLES DEL MODELO
# ============================================================

variables = [

    "edad",
    "antiguedad_meses",
    "ingreso_estimado",
    "num_lineas",
    "descuento_activo",

    "importe_total",
    "dias_retraso_pago",
    "impago_flag",

    "importe_mes_anterior",
    "consumo_mes_anterior",
    "retraso_mes_anterior",
    "impago_mes_anterior",

    "media_importe_3m",
    "media_retraso_3m",
    "impagos_3m",

    "cambio_consumo",
    "subida_factura",
    "variacion_consumo_pct",

    "stress_calidad_lag",
    "incidencia_masiva_lag",

    "cliente_moroso",
    "riesgo_financiero",
    "riesgo_consumo",
    "subida_factura_flag"

]

# ============================================================
# LIMPIEZA
# ============================================================

df = df.replace([np.inf, -np.inf], np.nan)

df = df.dropna(subset=["churn"])

for col in variables:
    df[col] = df[col].fillna(0)

X = df[variables]
y = df["churn"]

# ============================================================
# DISTRIBUCIÓN ORIGINAL
# ============================================================

print("="*60)
print("DISTRIBUCIÓN ORIGINAL CHURN")
print("="*60)
print(y.value_counts())
print(y.value_counts(normalize=True) * 100)

# ============================================================
# TRAIN / TEST
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ============================================================
# SMOTE SUAVE SOLO EN TRAIN
# ============================================================

smote = SMOTE(
    sampling_strategy=0.10,
    random_state=42,
    k_neighbors=5
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print("\n" + "="*60)
print("DISTRIBUCIÓN TRAIN DESPUÉS DE SMOTE")
print("="*60)
print(y_train_smote.value_counts())

# ============================================================
# MODELO RANDOM FOREST CONTROLADO
# ============================================================

modelo = RandomForestClassifier(
    n_estimators=400,
    max_depth=8,
    min_samples_split=30,
    min_samples_leaf=12,
    max_features="sqrt",
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

modelo.fit(X_train_smote, y_train_smote)

# ============================================================
# PROBABILIDADES
# ============================================================

y_prob = modelo.predict_proba(X_test)[:, 1]

# ============================================================
# THRESHOLD FIJO CONTROLADO
# ============================================================

threshold = 0.55

y_pred = (y_prob >= threshold).astype(int)

# ============================================================
# MÉTRICAS
# ============================================================

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc = roc_auc_score(y_test, y_prob)

# ============================================================
# RESULTADOS
# ============================================================

print("\n" + "="*60)
print("RESULTADOS MODELO FINAL")
print("="*60)

print(f"Threshold utilizado: {threshold}")
print(f"Accuracy: {accuracy:.2%}")
print(f"Precision churn: {precision:.4f}")
print(f"Recall churn: {recall:.4f}")
print(f"F1 churn: {f1:.4f}")
print(f"ROC-AUC: {roc:.4f}")

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

# ============================================================
# IMPORTANCIA VARIABLES
# ============================================================

importancias = pd.DataFrame({
    "Variable": variables,
    "Importancia": modelo.feature_importances_
}).sort_values(
    by="Importancia",
    ascending=False
)

print("\n" + "="*60)
print("IMPORTANCIA VARIABLES")
print("="*60)
print(importancias.head(20))

# ============================================================
# TOP 20 CLIENTES CON MAYOR RIESGO
# ============================================================

resultados = X_test.copy()
resultados["probabilidad_churn"] = y_prob
resultados["prediccion_churn"] = y_pred
resultados["churn_real"] = y_test.values

top_20 = resultados.sort_values(
    by="probabilidad_churn",
    ascending=False
).head(20)

print("\n" + "="*60)
print("TOP 20 CLIENTES CON MAYOR RIESGO DE CHURN")
print("="*60)
print(top_20)

NameError: name 'df_model' is not defined

In [14]:
# ============================================================
# MODELO CHURN MEJORADO CONTROLADO
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

from imblearn.over_sampling import SMOTE

# ============================================================
# CARGAR DATASET
# ============================================================

df = pd.read_csv("dataset_limpio_churn.csv")

# ============================================================
# VARIABLES DERIVADAS
# ============================================================

df["cliente_moroso"] = np.where(
    df["dias_retraso_pago"].fillna(0) > 0,
    1,
    0
)

df["riesgo_financiero"] = (
    df["impagos_3m"].fillna(0) * 5
    + df["media_retraso_3m"].fillna(0)
)

df["riesgo_consumo"] = (
    df["variacion_consumo_pct"].fillna(0).abs()
    + df["cambio_consumo"].fillna(0).abs()
)

df["subida_factura_flag"] = np.where(
    df["subida_factura"].fillna(0) > 10,
    1,
    0
)

# ============================================================
# VARIABLES DEL MODELO
# ============================================================

variables = [

    "edad",
    "antiguedad_meses",
    "ingreso_estimado",
    "num_lineas",
    "descuento_activo",

    "importe_total",
    "dias_retraso_pago",
    "impago_flag",

    "importe_mes_anterior",
    "consumo_mes_anterior",
    "retraso_mes_anterior",
    "impago_mes_anterior",

    "media_importe_3m",
    "media_retraso_3m",
    "impagos_3m",

    "cambio_consumo",
    "subida_factura",
    "variacion_consumo_pct",

    "stress_calidad_lag",
    "incidencia_masiva_lag",

    "cliente_moroso",
    "riesgo_financiero",
    "riesgo_consumo",
    "subida_factura_flag"

]

# ============================================================
# COMPROBAR QUE EXISTEN LAS COLUMNAS
# ============================================================

columnas_necesarias = variables + ["churn"]

columnas_faltantes = [
    col for col in columnas_necesarias 
    if col not in df.columns
]

if len(columnas_faltantes) > 0:
    print("ERROR: faltan estas columnas en el dataset:")
    print(columnas_faltantes)
    raise ValueError("Hay columnas que no existen en el dataset.")

# ============================================================
# LIMPIEZA
# ============================================================

df = df.replace([np.inf, -np.inf], np.nan)

df = df.dropna(subset=["churn"])

for col in variables:
    df[col] = df[col].fillna(0)

X = df[variables]
y = df["churn"].astype(int)

# ============================================================
# DISTRIBUCIÓN ORIGINAL
# ============================================================

print("="*60)
print("DISTRIBUCIÓN ORIGINAL CHURN")
print("="*60)
print(y.value_counts())
print(y.value_counts(normalize=True) * 100)

# ============================================================
# TRAIN / TEST
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ============================================================
# SMOTE SUAVE SOLO EN TRAIN
# ============================================================

smote = SMOTE(
    sampling_strategy=0.10,
    random_state=42,
    k_neighbors=5
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print("\n" + "="*60)
print("DISTRIBUCIÓN TRAIN DESPUÉS DE SMOTE")
print("="*60)
print(y_train_smote.value_counts())

# ============================================================
# MODELO RANDOM FOREST CONTROLADO
# ============================================================

modelo = RandomForestClassifier(
    n_estimators=400,
    max_depth=8,
    min_samples_split=30,
    min_samples_leaf=12,
    max_features="sqrt",
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

modelo.fit(X_train_smote, y_train_smote)

# ============================================================
# PROBABILIDADES
# ============================================================

y_prob = modelo.predict_proba(X_test)[:, 1]

# ============================================================
# THRESHOLD FIJO CONTROLADO
# ============================================================

threshold = 0.55

y_pred = (y_prob >= threshold).astype(int)

# ============================================================
# MÉTRICAS
# ============================================================

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc = roc_auc_score(y_test, y_prob)

# ============================================================
# RESULTADOS
# ============================================================

print("\n" + "="*60)
print("RESULTADOS MODELO FINAL")
print("="*60)

print(f"Threshold utilizado: {threshold}")
print(f"Accuracy: {accuracy:.2%}")
print(f"Precision churn: {precision:.4f}")
print(f"Recall churn: {recall:.4f}")
print(f"F1 churn: {f1:.4f}")
print(f"ROC-AUC: {roc:.4f}")

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

# ============================================================
# IMPORTANCIA VARIABLES
# ============================================================

importancias = pd.DataFrame({
    "Variable": variables,
    "Importancia": modelo.feature_importances_
}).sort_values(
    by="Importancia",
    ascending=False
)

print("\n" + "="*60)
print("IMPORTANCIA VARIABLES")
print("="*60)
print(importancias.head(20))

# ============================================================
# TOP 20 CLIENTES CON MAYOR RIESGO
# ============================================================

resultados = X_test.copy()
resultados["probabilidad_churn"] = y_prob
resultados["prediccion_churn"] = y_pred
resultados["churn_real"] = y_test.values

top_20 = resultados.sort_values(
    by="probabilidad_churn",
    ascending=False
).head(20)

print("\n" + "="*60)
print("TOP 20 CLIENTES CON MAYOR RIESGO DE CHURN")
print("="*60)
print(top_20)

KeyError: 'impagos_3m'